Để làm nhận diện khuôn mặt (Face Recognition) trên chip RV1106 (NPU 1 TOPS, RAM hạn chế như trên các mạch LuckFox Pico), một mình model YOLO thôi là chưa đủ.

Quy trình chuẩn trên RV1106 luôn cần 2 model .rknn chạy nối tiếp nhau (Pipeline):

Model 1 - Face Detection (Phát hiện khuôn mặt): Tìm vị trí khuôn mặt (bounding box) và 5 điểm mốc (mắt, mũi, miệng) để căn chỉnh (align) mặt cho thẳng.

Model khuyên dùng: RetinaFace (bản nhẹ mobilenet0.25, input 320x320 hoặc 640x640), YuNet (siêu nhẹ, rất hợp RV1106), hoặc YOLOv5-face / YOLOv8-face (bản nano n).


GitHub
Model 2 - Face Recognition / Feature Extraction (Trích xuất đặc trưng): Cắt ảnh khuôn mặt đã căn chỉnh (thường resize về 112x112), đưa qua model này để xuất ra một vector đặc trưng (embedding vector 128 hoặc 512 số), sau đó tính độ tương đồng (Cosine Similarity) với vector của người đã đăng ký trong database.   
EEWorld

Model khuyên dùng: MobileFaceNet (nhẹ nhất, chạy cực nhanh trên RV1106) hoặc ArcFace (ResNet18/MobileNet).


GitHub
Dưới đây là 2 cách để bạn có được bộ file .rknn này (từ nhanh nhất đến tự làm chủ động).

Cách 1: Tải sẵn hoặc dùng script chuẩn từ Rockchip / LuckFox (Khuyên dùng)
Rockchip đã tối ưu sẵn các model này trong kho mã nguồn chính thức rknn_model_zoo và wiki của LuckFox (hỗ trợ sẵn code C++ post-process đi kèm).   
LUCKFOX

Mở Google Colab, clone kho rknn_model_zoo của Rockchip và cài rknn-toolkit2:   
EEWorld

Bash
git clone https://github.com/airockchip/rknn_model_zoo.git
pip install rknn-toolkit2
Lấy model Detection (RetinaFace):

Bash
cd rknn_model_zoo/examples/RetinaFace/model
bash download_model.sh  # Tải sẵn file RetinaFace_mobile320.onnx
cd ../python
python convert.py ../model/RetinaFace_mobile320.onnx rv1106
→
→ Bạn sẽ thu được file RetinaFace.rknn (đã lượng tử hóa INT8 chuẩn cho RV1106).

Lấy model Recognition (MobileFaceNet / ArcFace):

Bạn có thể lấy file mobilefacenet.onnx từ bộ InsightFace (gói buffalo_sc hoặc buffalo_s, file tên là w600k_mbf.onnx - input 112x112).

Dùng chính script convert.py của rknn_model_zoo (hoặc script python bên dưới) để chuyển sang .rknn.

(Mẹo: Bạn cũng có thể tìm trên GitHub các repo như Luckfox-Pico hoặc junanxia/rkface để lấy cả bộ code C++ mẫu lẫn link tải sẵn 2 file retinaface.rknn và mobilefacenet.rknn).   
GitHub

Cách 2: Tự convert từ ONNX sang .rknn trên Google Colab
Nếu bạn đã có sẵn file .onnx (ví dụ yolov8n-face.onnx và w600k_mbf.onnx), bạn chạy đoạn code Python sau trên Google Colab để xuất ra file .rknn cho RV1106:

1. Chuẩn bị ảnh mẫu để lượng tử hóa (Quantization)
RV1106 chỉ chạy NPU ở định dạng INT8 nên bắt buộc phải có bước calibrate bằng ảnh thật (không dùng ảnh nhiễu ngẫu nhiên vì sẽ làm sai lệch vector nhận diện).

Tạo thư mục dataset_face/ chứa khoảng 30 - 50 ảnh khuôn mặt (đã crop vuông 112x112 cho model Recognition, hoặc ảnh cảnh người bình thường cho model Detection).

Tạo file dataset.txt liệt kê đường dẫn từng ảnh (mỗi dòng 1 đường dẫn).

2. Script Python convert cho RV1106
Python
from rknn.api import RKNN

def convert_onnx_to_rknn(onnx_path, rknn_path, dataset_txt, mean_values, std_values):
    rknn = RKNN(verbose=True)

    # 1. Cấu hình cho chip RV1106
    print('--> Config model')
    rknn.config(
        mean_values=mean_values,
        std_values=std_values,
        target_platform='rv1106'
    )

    # 2. Load ONNX
    print('--> Loading model')
    ret = rknn.load_onnx(model=onnx_path)
    if ret != 0:
        print('Load model failed!')
        return

    # 3. Build & Lượng tử hóa INT8 (Bắt buộc với RV1106)
    print('--> Building model')
    ret = rknn.build(do_quantization=True, dataset=dataset_txt)
    if ret != 0:
        print('Build model failed!')
        return

    # 4. Xuất file .rknn
    print('--> Export rknn model')
    ret = rknn.export_rknn(rknn_path)
    rknn.release()
    print('Done:', rknn_path)

# Ví dụ convert MobileFaceNet (w600k_mbf.onnx của InsightFace)
# Lưu ý: InsightFace chuẩn hóa ảnh về [-1, 1] nên mean=127.5, std=127.5
convert_onnx_to_rknn(
    onnx_path='w600k_mbf.onnx',
    rknn_path='mobilefacenet_rv1106.rknn',
    dataset_txt='dataset_112x112.txt',
    mean_values=[[127.5, 127.5, 127.5]],
    std_values=[[127.5, 127.5, 127.5]]
)
Lưu ý quan trọng để chạy mượt trên RV1106
Thông số mean_values và std_values: Phải khớp 100% với lúc train model gốc. Nếu sai thông số này, model Detection vẫn có thể khoanh được mặt nhưng model Recognition sẽ nhận diện nhầm người này sang người khác (vector bị sai lệch).

RetinaFace (rknn_model_zoo): mean=[[104, 117, 123]], std=[[1, 1, 1]]

MobileFaceNet (InsightFace): mean=[[127.5, 127.5, 127.5]], std=[[127.5, 127.5, 127.5]]

YOLO-face (Ultralytics): mean=[[0, 0, 0]], std=[[255, 255, 255]]

Giới hạn bộ nhớ (CMA RAM): Các bản RV1106 giá rẻ (như LuckFox Pico Mini/Plus) chỉ có 64MB hoặc 128MB RAM dùng chung cho cả Linux và NPU. Bạn nên chọn model thật nhẹ (RetinaFace 320x320 nặng ~2MB + MobileFaceNet nặng ~4MB) để load đồng thời cả 2 model vào RAM mà không bị lỗi tràn bộ nhớ (Out of memory).

Tải về máy: Sau khi chạy xong trên Colab, bạn mở tab Files (biểu tượng thư mục bên trái)
→
→ chuột phải vào file .rknn
→
→ chọn Download, sau đó đẩy vào board RV1106 qua adb push hoặc scp.

In [1]:
%%bash
set -e
cd /content

PY=/opt/rknn_env/bin/python

# 1. Cài uv và tạo môi trường Python 3.11
pip install -q uv
if [ ! -x "$PY" ]; then
  uv venv --python 3.11 /opt/rknn_env
fi

# 2. Cài rknn-toolkit2 và setuptools<81 (để có pkg_resources)
uv pip install --python "$PY" rknn-toolkit2 "setuptools<81"

# 3. Tải rknn_model_zoo và model RetinaFace_mobile320.onnx
if [ ! -d "/content/rknn_model_zoo" ]; then
  git clone --depth 1 https://github.com/airockchip/rknn_model_zoo.git
fi

cd /content/rknn_model_zoo/examples/RetinaFace/model
if [ ! -f "RetinaFace_mobile320.onnx" ]; then
  bash download_model.sh
fi

# 4. Convert sang .rknn cho chip rv1106
cd /content/rknn_model_zoo/examples/RetinaFace/python
"$PY" convert.py ../model/RetinaFace_mobile320.onnx rv1106

# 5. Kiểm tra file kết quả
ls -lh ../model/*.rknn

--> Config model
done
--> Loading model
done
--> Building model






compress=0, global_fuse=1, safe_fuse=0, output_optimize=1, op_group_sram_opt=0, op_group_nbuf_opt=0, enable_flash_attention=0, conv_eltwise_activation_fuse=1, enable_rnn_loop=1, memory_plan_strategy=BestFit, enable_core0_only_mode=0, disable_first_conv=0
done
--> Export rknn model
done
-rw-r--r-- 1 root root 1.7M Nov 24 03:34 ../model/RetinaFace.rknn


Using Python 3.11.18 environment at: /opt/rknn_env
Resolved 36 packages in 390ms
Prepared 1 package in 195ms
Installed 1 package in 8ms
 + setuptools==80.10.2
/opt/rknn_env/lib/python3.11/site-packages/rknn/api/rknn.py:51: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  self.rknn_base = RKNNBase(cur_path, verbose)
I rknn-toolkit2 version: 2.5.0
I OpFusing 2 : 100%|██████████████████████████████████████████████| 100/100 [00:01<00:00, 95.04it/s]
W build: found outlier value, this may affect quantization accuracy
                        const name          abs_mean    abs_std     outlier value
                        onnx::Conv_613      0.69        0.83        21.961      
I Quantize calibration include 1 samples.
I OpFusing 2 : 100%|████████████████████████████████████████████| 100/100 [00:00<00

Tải file .rknn về máy tính

Trên Colab, mở thanh Files (bên trái) -> đi tới /content/rknn_model_zoo/examples/RetinaFace/model/RetinaFace.rknn -> chuột phải chọn Download.

RetinaFace.rknn (1.6MB) đã lên board ngon lành. Bước tiếp theo là biên dịch 1 file binary C++ (rknn_retinaface_demo) trên Google Colab bằng đúng toolchain uClibc của Luckfox RV1106, rồi tải về thả vào thư mục colab/ để đẩy lên chạy NPU thật.

Dùng luôn source C++ RetinaFace của rknn_model_zoo vì nó đã có sẵn stb_image (đọc ảnh trực tiếp không cần OpenCV) và bộ hậu xử lý (anchor, box, 5 điểm mắt/mũi/miệng, NMS) khớp 100% với file RetinaFace.rknn.

1. Chạy cell này trên Google Colab để build binary C++
Mở 1 notebook Colab mới và chạy đoạn script sau:

Bash
%%bash
set -e
cd /content

# 1. Cài cmake nếu chưa có
apt-get update -qq && apt-get install -y -qq cmake git

# 2. Tải riêng toolchain uClibc của Luckfox RV1106 (dùng sparse-checkout cho nhẹ)
if [ ! -d "luckfox-pico" ]; then
  git clone --depth 1 --filter=blob:none --sparse https://github.com/LuckfoxTECH/luckfox-pico.git
  cd luckfox-pico
  git sparse-checkout set tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf
  cd /content
fi

# 3. Tải rknn_model_zoo
if [ ! -d "rknn_model_zoo" ]; then
  git clone --depth 1 https://github.com/airockchip/rknn_model_zoo.git
fi

# 4. Biên dịch RetinaFace cho RV1106 (armhf + uClibc)
export GCC_COMPILER=/content/luckfox-pico/tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf/bin/arm-rockchip830-linux-uclibcgnueabihf
cd /content/rknn_model_zoo
chmod +x build-linux.sh
./build-linux.sh -t rv1106 -a armhf -d RetinaFace

# 5. Kiểm tra file binary đầu ra
ls -lh /content/rknn_model_zoo/install/rv1106_linux_armhf/rknn_RetinaFace_demo/rknn_retinaface_demo
2. Tải binary về máy
Chạy thêm 1 cell Python nhỏ trên Colab để tải file về ngay:

Python
from google.colab import files

files.download(
    "/content/rknn_model_zoo/install/rv1106_linux_armhf/rknn_RetinaFace_demo/rknn_retinaface_demo"
)
Tải xong, chép file rknn_retinaface_demo vào thư mục colab/ trên Laptop (nằm cạnh RetinaFace.rknn).

Build xong và chép vào colab/ rồi thì báo mình — mình cập nhật luôn deploy.sh và luckfox_vision_node.py để đẩy binary lên và gọi NPU nhận diện khuôn mặt realtime từ luồng webcam PC.

In [3]:
%%bash
set -e
cd /content

# 1. Cài cmake nếu chưa có
apt-get update -qq && apt-get install -y -qq cmake git

# 2. Tải riêng toolchain uClibc của Luckfox RV1106 (dùng sparse-checkout cho nhẹ)
if [ ! -d "luckfox-pico" ]; then
  git clone --depth 1 --filter=blob:none --sparse https://github.com/LuckfoxTECH/luckfox-pico.git
  cd luckfox-pico
  git sparse-checkout set tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf
  cd /content
fi

# 3. Tải rknn_model_zoo
if [ ! -d "rknn_model_zoo" ]; then
  git clone --depth 1 https://github.com/airockchip/rknn_model_zoo.git
fi

# 4. Biên dịch RetinaFace cho RV1106 (armhf + uClibc)
export GCC_COMPILER=/content/luckfox-pico/tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf/bin/arm-rockchip830-linux-uclibcgnueabihf
cd /content/rknn_model_zoo
chmod +x build-linux.sh
./build-linux.sh -t rv1106 -a armhf -d RetinaFace

# 5. Kiểm tra file binary đầu ra
ls -lh /content/rknn_model_zoo/install/rv1106_linux_armhf/rknn_RetinaFace_demo/rknn_retinaface_demo

./build-linux.sh -t rv1106 -a armhf -d RetinaFace
/content/luckfox-pico/tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf/bin/arm-rockchip830-linux-uclibcgnueabihf
BUILD_DEMO_NAME=RetinaFace
BUILD_DEMO_PATH=examples/RetinaFace/cpp
TARGET_SOC=rv1106
TARGET_ARCH=armhf
BUILD_TYPE=Release
ENABLE_ASAN=OFF
DISABLE_RGA=OFF
DISABLE_LIBJPEG=OFF
INSTALL_DIR=/content/rknn_model_zoo/install/rv1106_linux_armhf/rknn_RetinaFace_demo
BUILD_DIR=/content/rknn_model_zoo/build/build_rknn_RetinaFace_demo_rv1106_linux_armhf_Release
CC=/content/luckfox-pico/tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf/bin/arm-rockchip830-linux-uclibcgnueabihf-gcc
CXX=/content/luckfox-pico/tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf/bin/arm-rockchip830-linux-uclibcgnueabihf-g++
-- The C compiler identification is GNU 8.3.0
-- The CXX compiler identification is GNU 8.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /content/lu

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into 'luckfox-pico'...
/content/rknn_model_zoo/utils/image_utils.c: In function 'write_image':
/content/rknn_model_zoo/utils/image_utils.c:290:35: warning: passing argument 1 of 'get_image_size' discards 'const' qualifier from pointer target type [-Wdiscarded-qualifiers]
         int size = get_image_size(img);
                                   ^~~
In file included from /content/rknn_model_zoo/utils/image_utils.c:19:
/content/rknn_model_zoo/utils/image_utils.h:67:36: note: expected 'image_buffer_t *' {aka 'struct <anonymous> *'} but argument is of type 'const image_buffer_t *' {aka 'const struct <anonymous> *'}
 int get_image_size(image_buffer_t* image);
                    ~~~~~~~~~~~~~~~~^~~~~
/content/rknn_model_zoo/utils/image_utils.c: In function 'convert_image_rga':
/content/rknn_model

In [4]:
from google.colab import files

files.download(
    "/content/rknn_model_zoo/install/rv1106_linux_armhf/rknn_RetinaFace_demo/rknn_retinaface_demo"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#---

In [5]:
%%bash
set -e
cat << 'EOF' > /content/retinaface_daemon.cc
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cmath>
#include <vector>
#include <algorithm>
#include <chrono>
#include <string>

#define STB_IMAGE_IMPLEMENTATION
#include "stb_image.h"
#include "rknn_api.h"

struct FaceBox {
    float x1, y1, x2, y2;
    float score;
    float lm[10];
};

static inline float fp16_to_fp32(uint16_t h) {
    uint32_t sign = (uint32_t)(h & 0x8000) << 16;
    uint32_t exp  = (h & 0x7C00) >> 10;
    uint32_t mant = (h & 0x03FF);
    uint32_t f;
    if (exp == 0) {
        if (mant == 0) f = sign;
        else {
            exp = 1;
            while ((mant & 0x0400) == 0) { mant <<= 1; exp--; }
            mant &= 0x03FF;
            f = sign | ((exp + 112) << 23) | (mant << 13);
        }
    } else if (exp == 31) {
        f = sign | 0x7F800000 | (mant << 13);
    } else {
        f = sign | ((exp + 112) << 23) | (mant << 13);
    }
    float res;
    memcpy(&res, &f, sizeof(res));
    return res;
}

static inline float get_val(const void* data, int idx, const rknn_tensor_attr& attr) {
    if (attr.type == RKNN_TENSOR_INT8) {
        return ((float)((const int8_t*)data)[idx] - (float)attr.zp) * attr.scale;
    } else if (attr.type == RKNN_TENSOR_UINT8) {
        return ((float)((const uint8_t*)data)[idx] - (float)attr.zp) * attr.scale;
    } else if (attr.type == RKNN_TENSOR_FLOAT16) {
        return fp16_to_fp32(((const uint16_t*)data)[idx]);
    } else {
        return ((const float*)data)[idx];
    }
}

static float iou(const FaceBox& a, const FaceBox& b) {
    float xx1 = std::max(a.x1, b.x1);
    float yy1 = std::max(a.y1, b.y1);
    float xx2 = std::min(a.x2, b.x2);
    float yy2 = std::min(a.y2, b.y2);
    float w = std::max(0.0f, xx2 - xx1);
    float h = std::max(0.0f, yy2 - yy1);
    float inter = w * h;
    float area_a = std::max(0.0f, a.x2 - a.x1) * std::max(0.0f, a.y2 - a.y1);
    float area_b = std::max(0.0f, b.x2 - b.x1) * std::max(0.0f, b.y2 - b.y1);
    return inter / (area_a + area_b - inter + 1e-6f);
}

struct Prior { float cx, cy, sx, sy; };

static std::vector<Prior> generate_priors(int img_w, int img_h) {
    std::vector<Prior> priors;
    const int steps[3] = {8, 16, 32};
    const int min_sizes[3][2] = {{16, 32}, {64, 128}, {256, 512}};
    for (int k = 0; k < 3; k++) {
        int fw = (int)ceil((float)img_w / steps[k]);
        int fh = (int)ceil((float)img_h / steps[k]);
        for (int i = 0; i < fh; i++) {
            for (int j = 0; j < fw; j++) {
                for (int m = 0; m < 2; m++) {
                    Prior p;
                    p.sx = (float)min_sizes[k][m] / img_w;
                    p.sy = (float)min_sizes[k][m] / img_h;
                    p.cx = (j + 0.5f) * steps[k] / img_w;
                    p.cy = (i + 0.5f) * steps[k] / img_h;
                    priors.push_back(p);
                }
            }
        }
    }
    return priors;
}

static void letterbox_rgb(const uint8_t* src, int sw, int sh,
                          uint8_t* dst, int dw, int dh,
                          float& scale, int& pad_x, int& pad_y) {
    scale = std::min((float)dw / sw, (float)dh / sh);
    int nw = (int)(sw * scale);
    int nh = (int)(sh * scale);
    pad_x = (dw - nw) / 2;
    pad_y = (dh - nh) / 2;
    memset(dst, 0, dw * dh * 3);
    for (int y = 0; y < nh; y++) {
        int sy = std::min(sh - 1, (int)(y / scale));
        uint8_t* drow = dst + ((y + pad_y) * dw + pad_x) * 3;
        const uint8_t* srow = src + sy * sw * 3;
        for (int x = 0; x < nw; x++) {
            int sx = std::min(sw - 1, (int)(x / scale));
            drow[x * 3 + 0] = srow[sx * 3 + 0];
            drow[x * 3 + 1] = srow[sx * 3 + 1];
            drow[x * 3 + 2] = srow[sx * 3 + 2];
        }
    }
}

int main(int argc, char** argv) {
    if (argc < 2) {
        fprintf(stderr, "Usage: %s <RetinaFace.rknn>\n", argv[0]);
        return 1;
    }
    FILE* fp = fopen(argv[1], "rb");
    if (!fp) { fprintf(stderr, "Cannot open model\n"); return 1; }
    fseek(fp, 0, SEEK_END);
    size_t model_size = ftell(fp);
    fseek(fp, 0, SEEK_SET);
    std::vector<uint8_t> model_data(model_size);
    if (fread(model_data.data(), 1, model_size, fp) != model_size) {
        fclose(fp);
        return 1;
    }
    fclose(fp);

    rknn_context ctx = 0;
    int ret = rknn_init(&ctx, model_data.data(), model_size, 0, nullptr);
    if (ret < 0) { fprintf(stderr, "rknn_init fail ret=%d\n", ret); return 1; }

    rknn_input_output_num io_num;
    rknn_query(ctx, RKNN_QUERY_IN_OUT_NUM, &io_num, sizeof(io_num));

    rknn_tensor_attr in_attr;
    memset(&in_attr, 0, sizeof(in_attr));
    in_attr.index = 0;
    rknn_query(ctx, RKNN_QUERY_NATIVE_INPUT_ATTR, &in_attr, sizeof(in_attr));
    in_attr.type = RKNN_TENSOR_UINT8;
    in_attr.fmt = RKNN_TENSOR_NHWC;
    int model_h = in_attr.dims[1];
    int model_w = in_attr.dims[2];

    rknn_tensor_mem* in_mem = rknn_create_mem(ctx, in_attr.size_with_stride);
    ret = rknn_set_io_mem(ctx, in_mem, &in_attr);
    if (ret < 0) { fprintf(stderr, "rknn_set_io_mem in fail ret=%d\n", ret); return 1; }

    std::vector<rknn_tensor_attr> out_attrs(io_num.n_output);
    std::vector<rknn_tensor_mem*> out_mems(io_num.n_output);
    for (uint32_t i = 0; i < io_num.n_output; i++) {
        memset(&out_attrs[i], 0, sizeof(rknn_tensor_attr));
        out_attrs[i].index = i;
        rknn_query(ctx, RKNN_QUERY_NATIVE_OUTPUT_ATTR, &out_attrs[i], sizeof(rknn_tensor_attr));
        out_mems[i] = rknn_create_mem(ctx, out_attrs[i].size_with_stride);
        ret = rknn_set_io_mem(ctx, out_mems[i], &out_attrs[i]);
        if (ret < 0) { fprintf(stderr, "rknn_set_io_mem out[%u] fail ret=%d\n", i, ret); return 1; }
    }

    auto priors = generate_priors(model_w, model_h);
    printf("READY\n");
    fflush(stdout);

    char line[512];
    while (fgets(line, sizeof(line), stdin)) {
        size_t len = strlen(line);
        while (len > 0 && (line[len - 1] == '\n' || line[len - 1] == '\r')) line[--len] = '\0';
        if (len == 0) continue;

        int sw = 0, sh = 0, sc = 0;
        uint8_t* img = stbi_load(line, &sw, &sh, &sc, 3);
        if (!img) {
            printf("{\"err\":\"stbi_load fail\",\"faces\":[]}\n");
            fflush(stdout);
            continue;
        }

        float scale = 1.0f;
        int pad_x = 0, pad_y = 0;
        letterbox_rgb(img, sw, sh, (uint8_t*)in_mem->virt_addr, model_w, model_h, scale, pad_x, pad_y);
        stbi_image_free(img);

        auto t0 = std::chrono::steady_clock::now();
        rknn_mem_sync(ctx, in_mem, RKNN_MEMORY_SYNC_TO_DEVICE);
        ret = rknn_run(ctx, nullptr);
        for (uint32_t i = 0; i < io_num.n_output; i++) {
            rknn_mem_sync(ctx, out_mems[i], RKNN_MEMORY_SYNC_FROM_DEVICE);
        }
        auto t1 = std::chrono::steady_clock::now();
        double npu_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();

        if (ret < 0) {
            printf("{\"err\":\"rknn_run fail %d\",\"npu_ms\":%.2f,\"faces\":[]}\n", ret, npu_ms);
            fflush(stdout);
            continue;
        }

        std::vector<FaceBox> cand;
        const float conf_thresh = 0.5f;
        const float nms_thresh = 0.4f;
        int n_priors = (int)priors.size();

        for (int i = 0; i < n_priors; i++) {
            float score = get_val(out_mems[1]->virt_addr, i * 2 + 1, out_attrs[1]);
            if (score < conf_thresh) continue;

            float loc0 = get_val(out_mems[0]->virt_addr, i * 4 + 0, out_attrs[0]);
            float loc1 = get_val(out_mems[0]->virt_addr, i * 4 + 1, out_attrs[0]);
            float loc2 = get_val(out_mems[0]->virt_addr, i * 4 + 2, out_attrs[0]);
            float loc3 = get_val(out_mems[0]->virt_addr, i * 4 + 3, out_attrs[0]);

            const Prior& p = priors[i];
            float cx = p.cx + loc0 * 0.1f * p.sx;
            float cy = p.cy + loc1 * 0.1f * p.sy;
            float bw = p.sx * expf(loc2 * 0.2f);
            float bh = p.sy * expf(loc3 * 0.2f);

            FaceBox fb;
            fb.x1 = ((cx - bw * 0.5f) * model_w - pad_x) / scale;
            fb.y1 = ((cy - bh * 0.5f) * model_h - pad_y) / scale;
            fb.x2 = ((cx + bw * 0.5f) * model_w - pad_x) / scale;
            fb.y2 = ((cy + bh * 0.5f) * model_h - pad_y) / scale;
            fb.score = score;

            for (int k = 0; k < 5; k++) {
                float lx = get_val(out_mems[2]->virt_addr, i * 10 + k * 2 + 0, out_attrs[2]);
                float ly = get_val(out_mems[2]->virt_addr, i * 10 + k * 2 + 1, out_attrs[2]);
                fb.lm[k * 2 + 0] = ((p.cx + lx * 0.1f * p.sx) * model_w - pad_x) / scale;
                fb.lm[k * 2 + 1] = ((p.cy + ly * 0.1f * p.sy) * model_h - pad_y) / scale;
            }
            cand.push_back(fb);
        }

        std::sort(cand.begin(), cand.end(), [](const FaceBox& a, const FaceBox& b) {
            return a.score > b.score;
        });

        std::vector<FaceBox> kept;
        for (const auto& c : cand) {
            bool suppress = false;
            for (const auto& k : kept) {
                if (iou(c, k) > nms_thresh) { suppress = true; break; }
            }
            if (!suppress) kept.push_back(c);
        }

        printf("{\"npu_ms\":%.2f,\"w\":%d,\"h\":%d,\"faces\":[", npu_ms, sw, sh);
        for (size_t i = 0; i < kept.size(); i++) {
            const auto& f = kept[i];
            if (i > 0) printf(",");
            printf("{\"score\":%.3f,\"box\":[%.1f,%.1f,%.1f,%.1f],\"lm\":[%.1f,%.1f,%.1f,%.1f,%.1f,%.1f,%.1f,%.1f,%.1f,%.1f]}",
                   f.score, f.x1, f.y1, f.x2, f.y2,
                   f.lm[0], f.lm[1], f.lm[2], f.lm[3], f.lm[4],
                   f.lm[5], f.lm[6], f.lm[7], f.lm[8], f.lm[9]);
        }
        printf("]}\n");
        fflush(stdout);
    }

    rknn_destroy_mem(ctx, in_mem);
    for (auto* m : out_mems) rknn_destroy_mem(ctx, m);
    rknn_destroy(ctx);
    return 0;
}
EOF

CXX=/content/luckfox-pico/tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf/bin/arm-rockchip830-linux-uclibcgnueabihf-g++
RKNN_INC=/content/rknn_model_zoo/3rdparty/rknpu2/include
RKNN_LIB=/content/rknn_model_zoo/3rdparty/rknpu2/Linux/armhf-uclibc
STB_INC=/content/rknn_model_zoo/3rdparty/stb_image

$CXX -O3 -std=c++11 /content/retinaface_daemon.cc \
  -I"$RKNN_INC" -I"$STB_INC" \
  -L"$RKNN_LIB" -lrknnmrt -lpthread -lm \
  -o /content/rknn_retinaface_daemon

ls -lh /content/rknn_retinaface_daemon

-rwxr-xr-x 1 root root 137K Nov 24 04:08 /content/rknn_retinaface_daemon


In [6]:
from google.colab import files
files.download("/content/rknn_retinaface_daemon")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#---

In [10]:
%%bash
set -e
/opt/rknn_env/bin/python - << 'EOF'
from rknn.api import RKNN

rknn = RKNN(verbose=False)
rknn.config(
    mean_values=[[127.5, 127.5, 127.5]],
    std_values=[[127.5, 127.5, 127.5]],
    target_platform="rv1106",
    quantized_dtype="asymmetric_quantized-8",
)
assert rknn.load_onnx(
    model="/content/mobilefacenet.onnx",
    inputs=["input.1"],
    input_size_list=[[1, 3, 112, 112]],
) == 0
assert rknn.build(do_quantization=True, dataset="/content/dataset.txt") == 0
assert rknn.export_rknn("/content/MobileFaceNet.rknn") == 0
rknn.release()
print("Convert OK -> /content/MobileFaceNet.rknn")
EOF

ls -lh /content/MobileFaceNet.rknn








compress=0, global_fuse=1, safe_fuse=0, output_optimize=1, op_group_sram_opt=0, op_group_nbuf_opt=0, enable_flash_attention=0, conv_eltwise_activation_fuse=1, enable_rnn_loop=1, memory_plan_strategy=BestFit, enable_core0_only_mode=0, disable_first_conv=0
Convert OK -> /content/MobileFaceNet.rknn
-rw-r--r-- 1 root root 3.6M Nov 24 05:01 /content/MobileFaceNet.rknn


/opt/rknn_env/lib/python3.11/site-packages/rknn/api/rknn.py:51: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  self.rknn_base = RKNNBase(cur_path, verbose)
I rknn-toolkit2 version: 2.5.0
W load_onnx: If you don't need to crop the model, don't set 'inputs'/'input_size_list'/'outputs'!
I OpFusing 2 : 100%|█████████████████████████████████████████████| 100/100 [00:00<00:00, 158.17it/s]
W build: found outlier value, this may affect quantization accuracy
                        const name       abs_mean    abs_std     outlier value
                        Conv_2#1_i1      0.01        0.15        18.241, -17.095
                        536              0.66        0.78        9.093       
                        563              0.57        0.68        -10.457     
                        617      

In [11]:
%%bash
set -e
cat << 'EOF' > /content/face_daemon.cc
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cmath>
#include <vector>
#include <algorithm>
#include <chrono>
#include <string>

#define STB_IMAGE_IMPLEMENTATION
#include "stb_image.h"
#include "rknn_api.h"

struct FaceBox {
    float x1, y1, x2, y2;
    float score;
    float lm[10];
    std::vector<float> emb;
};

struct RknnZeroCopy {
    rknn_context ctx = 0;
    rknn_input_output_num io_num{};
    rknn_tensor_attr in_attr{};
    rknn_tensor_mem* in_mem = nullptr;
    std::vector<rknn_tensor_attr> out_attrs;
    std::vector<rknn_tensor_mem*> out_mems;
    int model_w = 0, model_h = 0;

    bool init(const char* path) {
        FILE* fp = fopen(path, "rb");
        if (!fp) { fprintf(stderr, "Cannot open %s\n", path); return false; }
        fseek(fp, 0, SEEK_END);
        size_t sz = ftell(fp);
        fseek(fp, 0, SEEK_SET);
        std::vector<uint8_t> buf(sz);
        if (fread(buf.data(), 1, sz, fp) != sz) { fclose(fp); return false; }
        fclose(fp);

        int ret = rknn_init(&ctx, buf.data(), sz, 0, nullptr);
        if (ret < 0) { fprintf(stderr, "rknn_init %s fail %d\n", path, ret); return false; }

        rknn_query(ctx, RKNN_QUERY_IN_OUT_NUM, &io_num, sizeof(io_num));
        memset(&in_attr, 0, sizeof(in_attr));
        in_attr.index = 0;
        rknn_query(ctx, RKNN_QUERY_NATIVE_INPUT_ATTR, &in_attr, sizeof(in_attr));
        in_attr.type = RKNN_TENSOR_UINT8;
        in_attr.fmt = RKNN_TENSOR_NHWC;
        model_h = in_attr.dims[1];
        model_w = in_attr.dims[2];

        in_mem = rknn_create_mem(ctx, in_attr.size_with_stride);
        if (rknn_set_io_mem(ctx, in_mem, &in_attr) < 0) return false;

        out_attrs.resize(io_num.n_output);
        out_mems.resize(io_num.n_output);
        for (uint32_t i = 0; i < io_num.n_output; i++) {
            memset(&out_attrs[i], 0, sizeof(rknn_tensor_attr));
            out_attrs[i].index = i;
            rknn_query(ctx, RKNN_QUERY_NATIVE_OUTPUT_ATTR, &out_attrs[i], sizeof(rknn_tensor_attr));
            out_mems[i] = rknn_create_mem(ctx, out_attrs[i].size_with_stride);
            if (rknn_set_io_mem(ctx, out_mems[i], &out_attrs[i]) < 0) return false;
        }
        return true;
    }

    int run() {
        rknn_mem_sync(ctx, in_mem, RKNN_MEMORY_SYNC_TO_DEVICE);
        int ret = rknn_run(ctx, nullptr);
        for (uint32_t i = 0; i < io_num.n_output; i++) {
            rknn_mem_sync(ctx, out_mems[i], RKNN_MEMORY_SYNC_FROM_DEVICE);
        }
        return ret;
    }

    void destroy() {
        if (in_mem) rknn_destroy_mem(ctx, in_mem);
        for (auto* m : out_mems) if (m) rknn_destroy_mem(ctx, m);
        if (ctx) rknn_destroy(ctx);
    }
};

static inline float fp16_to_fp32(uint16_t h) {
    uint32_t sign = (uint32_t)(h & 0x8000) << 16;
    uint32_t exp  = (h & 0x7C00) >> 10;
    uint32_t mant = (h & 0x03FF);
    uint32_t f;
    if (exp == 0) {
        if (mant == 0) f = sign;
        else {
            exp = 1;
            while ((mant & 0x0400) == 0) { mant <<= 1; exp--; }
            mant &= 0x03FF;
            f = sign | ((exp + 112) << 23) | (mant << 13);
        }
    } else if (exp == 31) {
        f = sign | 0x7F800000 | (mant << 13);
    } else {
        f = sign | ((exp + 112) << 23) | (mant << 13);
    }
    float res;
    memcpy(&res, &f, sizeof(res));
    return res;
}

static inline float get_val(const void* data, int idx, const rknn_tensor_attr& attr) {
    if (attr.type == RKNN_TENSOR_INT8) return ((float)((const int8_t*)data)[idx] - (float)attr.zp) * attr.scale;
    if (attr.type == RKNN_TENSOR_UINT8) return ((float)((const uint8_t*)data)[idx] - (float)attr.zp) * attr.scale;
    if (attr.type == RKNN_TENSOR_FLOAT16) return fp16_to_fp32(((const uint16_t*)data)[idx]);
    return ((const float*)data)[idx];
}

static float iou(const FaceBox& a, const FaceBox& b) {
    float xx1 = std::max(a.x1, b.x1), yy1 = std::max(a.y1, b.y1);
    float xx2 = std::min(a.x2, b.x2), yy2 = std::min(a.y2, b.y2);
    float inter = std::max(0.0f, xx2 - xx1) * std::max(0.0f, yy2 - yy1);
    float area_a = std::max(0.0f, a.x2 - a.x1) * std::max(0.0f, a.y2 - a.y1);
    float area_b = std::max(0.0f, b.x2 - b.x1) * std::max(0.0f, b.y2 - b.y1);
    return inter / (area_a + area_b - inter + 1e-6f);
}

struct Prior { float cx, cy, sx, sy; };

static std::vector<Prior> generate_priors(int img_w, int img_h) {
    std::vector<Prior> priors;
    const int steps[3] = {8, 16, 32};
    const int min_sizes[3][2] = {{16, 32}, {64, 128}, {256, 512}};
    for (int k = 0; k < 3; k++) {
        int fw = (int)ceil((float)img_w / steps[k]);
        int fh = (int)ceil((float)img_h / steps[k]);
        for (int i = 0; i < fh; i++)
            for (int j = 0; j < fw; j++)
                for (int m = 0; m < 2; m++)
                    priors.push_back({(j + 0.5f) * steps[k] / img_w,
                                      (i + 0.5f) * steps[k] / img_h,
                                      (float)min_sizes[k][m] / img_w,
                                      (float)min_sizes[k][m] / img_h});
    }
    return priors;
}

static void letterbox_rgb(const uint8_t* src, int sw, int sh,
                          uint8_t* dst, int dw, int dh,
                          float& scale, int& pad_x, int& pad_y) {
    scale = std::min((float)dw / sw, (float)dh / sh);
    int nw = (int)(sw * scale), nh = (int)(sh * scale);
    pad_x = (dw - nw) / 2;
    pad_y = (dh - nh) / 2;
    memset(dst, 0, dw * dh * 3);
    for (int y = 0; y < nh; y++) {
        int sy = std::min(sh - 1, (int)(y / scale));
        uint8_t* drow = dst + ((y + pad_y) * dw + pad_x) * 3;
        const uint8_t* srow = src + sy * sw * 3;
        for (int x = 0; x < nw; x++) {
            int sx = std::min(sw - 1, (int)(x / scale));
            drow[x * 3 + 0] = srow[sx * 3 + 0];
            drow[x * 3 + 1] = srow[sx * 3 + 1];
            drow[x * 3 + 2] = srow[sx * 3 + 2];
        }
    }
}

static void align_face_112(const uint8_t* src, int sw, int sh,
                           const float lm[10], uint8_t* dst, int dw, int dh) {
    static const float ref[5][2] = {
        {38.2946f, 51.6963f}, {73.5318f, 51.5014f},
        {56.0252f, 71.7366f}, {41.5493f, 92.3655f}, {70.7299f, 92.2041f}
    };
    float scale_x = (float)dw / 112.0f, scale_y = (float)dh / 112.0f;
    float mxd = 0, myd = 0, mxs = 0, mys = 0;
    for (int i = 0; i < 5; i++) {
        mxd += ref[i][0] * scale_x; myd += ref[i][1] * scale_y;
        mxs += lm[i * 2];           mys += lm[i * 2 + 1];
    }
    mxd /= 5.0f; myd /= 5.0f; mxs /= 5.0f; mys /= 5.0f;

    float num_a = 0, num_b = 0, denom = 0;
    for (int i = 0; i < 5; i++) {
        float dx = ref[i][0] * scale_x - mxd;
        float dy = ref[i][1] * scale_y - myd;
        float sx = lm[i * 2] - mxs;
        float sy = lm[i * 2 + 1] - mys;
        num_a += dx * sx + dy * sy;
        num_b += dx * sy - dy * sx;
        denom += dx * dx + dy * dy;
    }
    float a = num_a / (denom + 1e-6f);
    float b = num_b / (denom + 1e-6f);
    float tx = mxs - (a * mxd - b * myd);
    float ty = mys - (b * mxd + a * myd);

    for (int y = 0; y < dh; y++) {
        for (int x = 0; x < dw; x++) {
            float sx = a * x - b * y + tx;
            float sy = b * x + a * y + ty;
            uint8_t* out = dst + (y * dw + x) * 3;
            int x0 = (int)floorf(sx), y0 = (int)floorf(sy);
            if (x0 < 0 || y0 < 0 || x0 + 1 >= sw || y0 + 1 >= sh) {
                out[0] = out[1] = out[2] = 0;
                continue;
            }
            float fx = sx - x0, fy = sy - y0;
            const uint8_t* p00 = src + (y0 * sw + x0) * 3;
            const uint8_t* p10 = p00 + 3;
            const uint8_t* p01 = p00 + sw * 3;
            const uint8_t* p11 = p01 + 3;
            for (int c = 0; c < 3; c++) {
                float v = (1 - fx) * (1 - fy) * p00[c] + fx * (1 - fy) * p10[c]
                        + (1 - fx) * fy * p01[c]       + fx * fy * p11[c];
                out[c] = (uint8_t)std::min(255.0f, std::max(0.0f, v));
            }
        }
    }
}

int main(int argc, char** argv) {
    if (argc < 3) {
        fprintf(stderr, "Usage: %s <RetinaFace.rknn> <MobileFaceNet.rknn>\n", argv[0]);
        return 1;
    }
    RknnZeroCopy det, rec;
    if (!det.init(argv[1]) || !rec.init(argv[2])) return 1;

    auto priors = generate_priors(det.model_w, det.model_h);
    int emb_dim = rec.out_attrs[0].n_elems;

    printf("READY\n");
    fflush(stdout);

    char line[512];
    while (fgets(line, sizeof(line), stdin)) {
        size_t len = strlen(line);
        while (len > 0 && (line[len - 1] == '\n' || line[len - 1] == '\r')) line[--len] = '\0';
        if (len == 0) continue;

        int sw = 0, sh = 0, sc = 0;
        uint8_t* img = stbi_load(line, &sw, &sh, &sc, 3);
        if (!img) {
            printf("{\"err\":\"stbi_load fail\",\"faces\":[]}\n");
            fflush(stdout);
            continue;
        }

        float scale = 1.0f;
        int pad_x = 0, pad_y = 0;
        letterbox_rgb(img, sw, sh, (uint8_t*)det.in_mem->virt_addr, det.model_w, det.model_h, scale, pad_x, pad_y);

        auto t0 = std::chrono::steady_clock::now();
        int ret = det.run();
        auto t1 = std::chrono::steady_clock::now();
        double det_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();

        std::vector<FaceBox> cand;
        if (ret >= 0) {
            for (int i = 0; i < (int)priors.size(); i++) {
                float score = get_val(det.out_mems[1]->virt_addr, i * 2 + 1, det.out_attrs[1]);
                if (score < 0.5f) continue;

                float l0 = get_val(det.out_mems[0]->virt_addr, i * 4 + 0, det.out_attrs[0]);
                float l1 = get_val(det.out_mems[0]->virt_addr, i * 4 + 1, det.out_attrs[0]);
                float l2 = get_val(det.out_mems[0]->virt_addr, i * 4 + 2, det.out_attrs[0]);
                float l3 = get_val(det.out_mems[0]->virt_addr, i * 4 + 3, det.out_attrs[0]);
                const Prior& p = priors[i];
                float cx = p.cx + l0 * 0.1f * p.sx, cy = p.cy + l1 * 0.1f * p.sy;
                float bw = p.sx * expf(l2 * 0.2f),  bh = p.sy * expf(l3 * 0.2f);

                FaceBox fb;
                fb.x1 = ((cx - bw * 0.5f) * det.model_w - pad_x) / scale;
                fb.y1 = ((cy - bh * 0.5f) * det.model_h - pad_y) / scale;
                fb.x2 = ((cx + bw * 0.5f) * det.model_w - pad_x) / scale;
                fb.y2 = ((cy + bh * 0.5f) * det.model_h - pad_y) / scale;
                fb.score = score;
                for (int k = 0; k < 5; k++) {
                    float lx = get_val(det.out_mems[2]->virt_addr, i * 10 + k * 2 + 0, det.out_attrs[2]);
                    float ly = get_val(det.out_mems[2]->virt_addr, i * 10 + k * 2 + 1, det.out_attrs[2]);
                    fb.lm[k * 2 + 0] = ((p.cx + lx * 0.1f * p.sx) * det.model_w - pad_x) / scale;
                    fb.lm[k * 2 + 1] = ((p.cy + ly * 0.1f * p.sy) * det.model_h - pad_y) / scale;
                }
                cand.push_back(fb);
            }
        }

        std::sort(cand.begin(), cand.end(), [](const FaceBox& a, const FaceBox& b) { return a.score > b.score; });
        std::vector<FaceBox> kept;
        for (const auto& c : cand) {
            bool sup = false;
            for (const auto& k : kept) if (iou(c, k) > 0.4f) { sup = true; break; }
            if (!sup) kept.push_back(c);
            if (kept.size() >= 3) break;
        }

        double rec_ms = 0.0;
        for (auto& f : kept) {
            align_face_112(img, sw, sh, f.lm, (uint8_t*)rec.in_mem->virt_addr, rec.model_w, rec.model_h);
            auto tr0 = std::chrono::steady_clock::now();
            int rret = rec.run();
            auto tr1 = std::chrono::steady_clock::now();
            rec_ms += std::chrono::duration<double, std::milli>(tr1 - tr0).count();

            if (rret >= 0) {
                f.emb.resize(emb_dim);
                float norm = 0.0f;
                for (int d = 0; d < emb_dim; d++) {
                    float v = get_val(rec.out_mems[0]->virt_addr, d, rec.out_attrs[0]);
                    f.emb[d] = v;
                    norm += v * v;
                }
                norm = sqrtf(norm) + 1e-6f;
                for (int d = 0; d < emb_dim; d++) f.emb[d] /= norm;
            }
        }
        stbi_image_free(img);

        printf("{\"npu_ms\":%.2f,\"det_ms\":%.2f,\"rec_ms\":%.2f,\"w\":%d,\"h\":%d,\"faces\":[",
               det_ms + rec_ms, det_ms, rec_ms, sw, sh);
        for (size_t i = 0; i < kept.size(); i++) {
            const auto& f = kept[i];
            if (i > 0) printf(",");
            printf("{\"score\":%.3f,\"box\":[%.1f,%.1f,%.1f,%.1f],\"lm\":[%.1f,%.1f,%.1f,%.1f,%.1f,%.1f,%.1f,%.1f,%.1f,%.1f],\"emb\":[",
                   f.score, f.x1, f.y1, f.x2, f.y2,
                   f.lm[0], f.lm[1], f.lm[2], f.lm[3], f.lm[4],
                   f.lm[5], f.lm[6], f.lm[7], f.lm[8], f.lm[9]);
            for (size_t d = 0; d < f.emb.size(); d++) {
                if (d > 0) printf(",");
                printf("%.4f", f.emb[d]);
            }
            printf("]}");
        }
        printf("]}\n");
        fflush(stdout);
    }

    det.destroy();
    rec.destroy();
    return 0;
}
EOF

CXX=/content/luckfox-pico/tools/linux/toolchain/arm-rockchip830-linux-uclibcgnueabihf/bin/arm-rockchip830-linux-uclibcgnueabihf-g++
RKNN_INC=/content/rknn_model_zoo/3rdparty/rknpu2/include
RKNN_LIB=/content/rknn_model_zoo/3rdparty/rknpu2/Linux/armhf-uclibc
STB_INC=/content/rknn_model_zoo/3rdparty/stb_image

$CXX -O3 -std=c++11 /content/face_daemon.cc \
  -I"$RKNN_INC" -I"$STB_INC" \
  -L"$RKNN_LIB" -lrknnmrt -lpthread -lm \
  -o /content/rknn_face_daemon

ls -lh /content/rknn_face_daemon

-rwxr-xr-x 1 root root 142K Nov 24 05:03 /content/rknn_face_daemon


In [12]:
from google.colab import files
files.download("/content/MobileFaceNet.rknn")
files.download("/content/rknn_face_daemon")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>